# Oshiokeme Williams – Exploratory Data Analysis (EDA) & Seasonality Visualizations
**Project:** Food Price Prediction in Nigeria: Informing Consumer Budgeting through Data Analysis using Machine Learning \
**Deliverable:** Jupyter Notebook (`notebooks/03_eda_seasonality.ipynb`) & Saved Chart Images (`outputs/charts/`)

## 1. Environment Setup & Data Ingestion
In this section, we import necessary libraries (`pandas`, `numpy`, `matplotlib`, `seaborn`), load our cleaned microdata (`cleaned_foodprice.csv`), and verify data types and date indexing.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates

# Create output directories if they don't exist
os.makedirs('../outputs/charts', exist_ok=True)

# Styling setup
sns.set_style("whitegrid")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

# Load dataset
df = pd.read_csv('../data/cleaned_foodprice.csv')
df['Date'] = pd.to_datetime(df['Year'].astype(str) + '-' + df['Month_Num'].astype(str).str.zfill(2) + '-01')
df = df.sort_values(['Item Label', 'Date']).reset_index(drop=True)

print(f"Dataset Shape: {df.shape}")
display(df.head())

## 2. Summary Statistics & Cumulative Price Change
We compute summary statistics (mean, median, standard deviation, min, max) and percentage changes from **January 2025 to May 2026** across all 14 staple food items to identify inflation spikes and structural trends.

In [ ]:
# Compute percentage change from Jan 2025 to May 2026
jan25 = df[df['Date'] == df['Date'].min()].set_index('Item Label')['Avg_Price']
may26 = df[df['Date'] == df['Date'].max()].set_index('Item Label')['Avg_Price']
pct_change = ((may26 - jan25) / jan25 * 100).sort_values(ascending=False)

# Summary statistics table
summary = df.groupby('Item Label')['Avg_Price'].agg(
    Count='count',
    Mean='mean',
    Median='median',
    Min='min',
    Max='max',
    Std='std'
).round(2)
summary['Pct_Change_Jan25_May26'] = pct_change.round(2)

display(summary.sort_values('Pct_Change_Jan25_May26', ascending=False))

## 3. Historical Price Trend Visualizations (Small Multiples)
We plot historical price trends (`Date` vs. `Avg_Price`) from January 2025 to May 2026 for each staple food item, highlighting peak price points and classifying them by inflation severity.

In [ ]:
ordered_items = pct_change.index.tolist()

fig, axes = plt.subplots(4, 4, figsize=(18, 13), sharex=True)
axes = axes.flatten()

for idx, item in enumerate(ordered_items):
    ax = axes[idx]
    sub = df[df['Item Label'] == item]
    chg = pct_change[item]
    color = '#D9534F' if chg > 5 else ('#2E8B57' if chg < -5 else '#4682B4')
    
    ax.plot(sub['Date'], sub['Avg_Price'], marker='o', markersize=4, linewidth=2, color=color)
    ax.fill_between(sub['Date'], sub['Avg_Price'], sub['Avg_Price'].min()*0.95, color=color, alpha=0.08)
    
    # Peak marker
    peak_row = sub.loc[sub['Avg_Price'].idxmax()]
    ax.scatter(peak_row['Date'], peak_row['Avg_Price'], color='#111111', zorder=5, s=30)
    
    sign = '+' if chg >= 0 else ''
    ax.set_title(f"{item}\n({sign}{chg:.1f}% Jan '25–May '26)", fontsize=10, fontweight='bold', pad=8)
    ax.set_ylabel("Avg Price (₦)", fontsize=8.5)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
    ax.tick_params(axis='x', rotation=30, labelsize=8)
    ax.grid(True, linestyle='--', alpha=0.4)

# Hide unused axes
for j in range(len(ordered_items), len(axes)):
    axes[j].axis('off')

fig.suptitle("Nigeria Staple Food Historical Price Trends (Jan 2025 – May 2026)", fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.savefig('../outputs/charts/01_historical_price_trends.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Seasonality Analysis (Grouping by Month_Num across Years)
To address lecturer feedback on seasonality, we group data by `Month_Num` (Jan–Dec) across years to compute a **Seasonal Price Index** ($	ext{Monthly Mean} / 	ext{Item Annual Mean} 	imes 100$). This isolates lean/planting season surges from harvest relief.

In [ ]:
# Pivot table for Month_Num vs Item Label
pivot_month = df.pivot_table(index='Item Label', columns='Month_Num', values='Avg_Price', aggfunc='mean')
seasonal_index = pivot_month.div(pivot_month.mean(axis=1), axis=0) * 100

month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
seasonal_index.columns = month_names

# Sort by lean season peak (June-July intensity)
seasonal_index['Lean_Peak'] = seasonal_index[['Jun', 'Jul']].mean(axis=1)
seasonal_index = seasonal_index.sort_values('Lean_Peak', ascending=False).drop(columns='Lean_Peak')

print("Seasonal Price Index (100 = Annual Item Average):")
display(seasonal_index.round(1))

### Seasonality Visualizations
We generate both a **Seasonal Heatmap** and a **Composite Seasonal & MoM Trigger Chart** to visualize planting vs. harvest vs. festive periods.

In [ ]:
# Chart 2: Seasonality Heatmap
fig, ax = plt.subplots(figsize=(13, 8))
sns.heatmap(seasonal_index, annot=True, fmt=".1f", cmap="RdYlGn_r", center=100, linewidths=0.6,
            cbar_kws={'label': 'Seasonal Price Index (100 = Annual Average)'}, ax=ax)
ax.set_title("Seasonal Price Index Heatmap Across Months (Month_Num: Jan–Dec)\nValues > 100 Indicate Seasonal Price Surges; Values < 100 Indicate Harvest Relief",
             fontsize=12.5, fontweight='bold', pad=14)
ax.set_xlabel("Calendar Month", fontsize=11)
ax.set_ylabel("Staple Food Item", fontsize=11)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('../outputs/charts/03_seasonality_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Chart 3: Composite Seasonal Index & Month-over-Month (MoM) Triggers
monthly_mom = df.groupby('Month_Num')['MoM'].mean().sort_index().values
avg_seasonal_idx = seasonal_index.mean(axis=0).values

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8.5), sharex=True)

# Top Panel: Composite Seasonal Index
colors_idx = ['#D9534F' if v > 101 else ('#2E8B57' if v < 98.5 else '#5BC0DE') for v in avg_seasonal_idx]
bars1 = ax1.bar(month_names, avg_seasonal_idx, color=colors_idx, edgecolor='#333333', linewidth=0.6)
ax1.axhline(100, color='black', linestyle='--', linewidth=1)
ax1.set_ylim(90, 112)
ax1.set_ylabel("Composite Seasonal Index\n(100 = Annual Mean)", fontsize=10.5)
ax1.set_title("Composite Seasonal Price Cycle & Month-over-Month (MoM) Triggers (Jan–Dec)", fontsize=13, fontweight='bold', pad=12)

for bar, val in zip(bars1, avg_seasonal_idx):
    ax1.text(bar.get_x() + bar.get_width()/2, val + 0.4, f"{val:.1f}", ha='center', va='bottom', fontsize=9, fontweight='bold')

# Bottom Panel: Average MoM % Change
colors_mom = ['#D9534F' if v > 0 else '#2E8B57' for v in monthly_mom]
bars2 = ax2.bar(month_names, monthly_mom, color=colors_mom, edgecolor='#333333', linewidth=0.6)
ax2.axhline(0, color='black', linewidth=1)
ax2.set_ylabel("Mean MoM Price Change (%)", fontsize=10.5)
ax2.set_xlabel("Month (Grouped by Month_Num: 1 = Jan to 12 = Dec)", fontsize=11)
ax2.set_ylim(-11, 5.5)

for bar, val in zip(bars2, monthly_mom):
    offset = 0.3 if val >= 0 else -0.8
    ax2.text(bar.get_x() + bar.get_width()/2, val + offset, f"{val:+.1f}%", ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/charts/04_seasonal_mom_triggers.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Conclusion & Deliverable Summary
This EDA and seasonality analysis confirms:
1. **Planting/Lean Season (May–July):** Consistently triggers sharp price surges across tubers, grains, and vegetables due to wet-season supply bottlenecks.
2. **Harvest Relief (August–November):** Fresh harvest arrivals drive down staple prices significantly (e.g., Garri, Beans, Yam).
3. **December/January Trough:** Bulk commodities experience dry-season harvest dips, while protein and processed items remain sticky.